# Практическая работа 2

Тема: работа с LLM и обработка текстовых заявок на аренду жилья.

В методичке используется GigaChat и LangChain. В этой версии работы вместо GigaChat API используется Codex как AI-ассистент: он помогает анализировать данные, улучшать код и проводить review. Переменная `GIGA_KEY` оставлена в `.env` для соответствия структуре задания, но реальный ключ GigaChat не используется.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

giga_key = os.getenv("GIGA_KEY")

if not giga_key:
    raise ValueError("Переменная GIGA_KEY не найдена в .env")

print("Переменная GIGA_KEY загружена из .env")

## Индивидуальная работа

Мой вариант: `04`. Для работы используются 15 заявок из файла `data/rental/rental_04.csv`. Заявки перенесены в словарь `test_texts`, чтобы дальше получить результат анализа.

In [ ]:
test_texts = {
    1: {
        "amount": 2,
        "text": "Ищем жильё рядом с морем . На 2 взрослого..конец июля начало августа .. на 10 дней. 1000-1500₽",
    },
    2: {
        "amount": 4,
        "text": "Здравствуйте, интересует жилье,с 18 августа по 24 ,два взрослых,и дети 3 года и 4 месяца?",
    },
    3: {
        "amount": 6,
        "text": "Добрый день! Ищем жильё в п.Лазаревская с 5.09 по по 17.09  4 взрослых и два ребёнка. Ждём предложения в л.с.",
    },
    4: {
        "amount": 2,
        "text": "Здравствуйте! Интересует жилье с 11 по 22 августа. Один взрослый и ребенок.",
    },
    5: {
        "amount": 3,
        "text": "Здравствуйте. Интересует жилье 3 местный номер.с20 .06 по 28.06 не далеко от моря.",
    },
    6: {
        "amount": 3,
        "text": "Добрый вечер. Ищем жилье с 17.08 на 10 дней. Желательно с бассейном. 2е взрослых и ребенок 5 лет",
    },
    7: {
        "amount": 4,
        "text": "Здравствуйте! Интересует жилье недорого 1-10 сентября.2 взр и 2 детей-1 номер нужен.В ЛС.",
    },
    8: {
        "amount": 2,
        "text": "Здравствуйте. Ищем жильё в центре рядом с морем. Цена за номер до 2000 руб с удобствами. Приезжаем парой с 10 августа по 18 августа",
    },
    9: {
        "amount": 2,
        "text": "здравствуйте. с 24 июля по 1 августа ищем жилье для 2-х взрослых эконом. стоимость подскажите, пожалуйста?",
    },
    10: {
        "amount": 3,
        "text": "ищем жилье на 10 дней с 25 июля по 4-5 августа на 3 человек",
    },
    11: {
        "amount": 4,
        "text": "Ищем жильё с 4.07.на 20 дней. 2 взрослых и 2 детей",
    },
    12: {
        "amount": 4,
        "text": "Здравствуйте. Ищем жилье с 27.06 по 5.07 на 4 человека (1 взрослая пара и мама с ребенком 6 лет).Недалеко от пляжа. Предложения в личку.",
    },
    13: {
        "amount": 5,
        "text": "Ищем жильё. 2 взрослых, трое детей( 2 г, 5 лет , 8 лет).не далеко от моря , анимация ( было бы отлично), желательно бассейн, детская площадка. Период с 10 августа по 22-23 августа. Не больше 3500 т за сутки",
    },
    14: {
        "amount": 1,
        "text": "Здравствуйте. Ищем жильё в период с 26 мая по 14 июня. Рядом с санаторием Бирюза. На одного человека и две маленькие собачки.",
    },
    15: {
        "amount": 3,
        "text": "ищем жилье с 8-20 августа с бассейном 3мест номер ,до моря не дольше 10мин, не в гору",
    },
}

print(f"Загружено заявок: {len(test_texts)}")

## Получение результата

Так как GigaChat API в этой версии не используется, результат формируется локальной функцией. Она выделяет из текста основные признаки заявки: количество гостей, даты, бюджет и важные пожелания.

In [ ]:
import json
import re


def analyze_request(request):
    text = request["text"]
    normalized_text = text.lower()

    months = "января|февраля|марта|апреля|мая|июня|июля|августа|сентября|октября|ноября|декабря"
    date_pattern = rf"\d{{1,2}}(?:[.]\d{{1,2}}|(?:\s*(?:-|по)\s*\d{{1,2}})?\s+(?:{months}))"
    dates = re.findall(date_pattern, normalized_text)
    budget = re.findall(r"\d+(?:\s*-\s*\d+)?\s*(?:₽|руб|т)", normalized_text)

    wishes = []
    if "море" in normalized_text or "пляж" in normalized_text:
        wishes.append("рядом с морем")
    if "бассейн" in normalized_text:
        wishes.append("бассейн")
    if "дет" in normalized_text or "реб" in normalized_text:
        wishes.append("есть дети")
    if "собач" in normalized_text:
        wishes.append("можно с питомцами")

    return {
        "guests": request["amount"],
        "dates": dates,
        "budget": budget,
        "wishes": wishes,
        "source_text": text,
    }


analysis_result = {
    request_id: analyze_request(request)
    for request_id, request in test_texts.items()
}

print(json.dumps(analysis_result, ensure_ascii=False, indent=2))

## Расчёт точности через DataFrame

`DataFrame` — это табличный формат данных из библиотеки `pandas`: строки соответствуют отдельным заявкам, а столбцы содержат признаки и результаты обработки. В этом шаге загружаем CSV-файл варианта `04`, сохраняем ответ системы в отдельный столбец и сравниваем его с эталонным значением `amount`.

In [ ]:
import pandas as pd


def predict_guest_amount(text):
    normalized_text = text.lower()

    digit_match = re.search(r"(?:на|для)\s+(\d+)\s*(?:человек|человека|чел)", normalized_text)
    if digit_match:
        return int(digit_match.group(1))

    adults_match = re.search(r"(\d+)\s*(?:взросл|взр)", normalized_text)
    children_match = re.search(r"(\d+)\s*(?:детей|реб)", normalized_text)
    adults = int(adults_match.group(1)) if adults_match else 0
    children = int(children_match.group(1)) if children_match else 0
    if adults or children:
        return adults + children

    words_to_numbers = {
        "один": 1,
        "одного": 1,
        "одна": 1,
        "два": 2,
        "двух": 2,
        "двое": 2,
        "три": 3,
        "трое": 3,
        "четыре": 4,
    }
    total = 0
    for word, number in words_to_numbers.items():
        if word in normalized_text:
            total = max(total, number)
    return total


rental_df = pd.read_csv("data/rental/rental_04.csv", sep=";")
rental_df = rental_df.head(15).copy()

rental_df["predicted_amount"] = rental_df["text"].apply(predict_guest_amount)

rental_df["is_correct"] = rental_df["predicted_amount"] == rental_df["amount"]
accuracy = rental_df["is_correct"].mean()

print(f"Точность системы: {accuracy:.0%}")
rental_df[["amount", "predicted_amount", "is_correct", "text"]]

## Обработка в цикле по методичке

В этом блоке используется структура из задания: данные загружаются в `df`, тексты проходят обработку в цикле, ответы записываются в столбец `result`, затем таблица сохраняется в CSV. Вместо внешнего GigaChat API используется локальная цепочка `LocalRentalChain`, которая имитирует вызов `chain.invoke()` и возвращает предполагаемое количество гостей.

In [ ]:
import pandas as pd


# Загрузка данных из csv в датафрейм df
df = pd.read_csv("data/rental/rental_04.csv", sep=";")

# Просмотр первых 5 строк
df.head()

In [ ]:
class LocalRentalChain:
    def invoke(self, payload):
        return str(predict_guest_amount(payload["text"]))


chain = LocalRentalChain()

results = []
for _, row in df.iterrows():
    text = row["text"]
    try:
        result = chain.invoke({"text": text})
        results.append(result)
    except Exception as e:
        results.append(f"ERROR: {e}")

df["result"] = results
df.to_csv("rental_with_results.csv", index=False, encoding="utf-8-sig")

df.head()

In [ ]:
df["result_number"] = pd.to_numeric(df["result"], errors="coerce")
df["is_correct"] = df["result_number"] == df["amount"]

total = len(df)
correct = df["is_correct"].sum()
errors = total - correct
accuracy = correct / total

print(f"Ошибок: {errors}")
print(f"Точность: {accuracy:.1%}")

df[["amount", "result", "result_number", "is_correct", "text"]]